In [74]:
import pandas as pd
import sqlite3 as sql

In [76]:
con = sql.connect('superstore.db')

In [78]:
df = pd.read_excel('superstore dataset.xlsx')
df.head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [79]:
df.to_sql('orders', con, if_exists='replace', index=False)

9994

In [82]:
pd.read_sql_query("SELECT * FROM orders LIMIT 5;", con)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12 00:00:00,2016-06-16 00:00:00,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Monthly Revenue and Order Volume

In [85]:
query = """
SELECT 
    STRFTIME('%Y', [Order Date]) AS Year,
    STRFTIME('%m', [Order Date]) AS Month,
    SUM(Sales) AS Total_Revenue,
    COUNT(DISTINCT [Order ID]) AS Total_Orders
FROM orders
GROUP BY Year, Month
ORDER BY Year, Month
"""
monthly_summary = pd.read_sql_query(query, con)
monthly_summary

,Year,Month,Total_Revenue,Total_Orders
0,2014,01,14236.8950,32
1,2014,02,4519.8920,28
2,2014,03,55691.0090,71
3,2014,04,28295.3450,66
4,2014,05,23648.2870,69
5,2014,06,34595.1276,66
6,2014,07,33946.3930,65
7,2014,08,27909.4685,72
8,2014,09,81777.3508,130
9,2014,10,31453.3930,78


## Total Revenue and Order Volume by Year

In [88]:
query1 = """
SELECT 
    STRFTIME('%Y', [Order Date]) AS Year,
    SUM(Sales) AS Total_Revenue,
    COUNT(DISTINCT [Order ID]) AS Total_Orders
FROM orders
GROUP BY Year
ORDER BY Year
"""
yearly_summary = pd.read_sql_query(query1, con)
yearly_summary


,Year,Total_Revenue,Total_Orders
0,2014,484247.4981,969
1,2015,470532.5090,1038
2,2016,609205.5980,1315
3,2017,733215.2552,1687


## Average Order Value by Month

In [91]:
query2 = """
SELECT 
    STRFTIME('%Y', [Order Date]) AS Year,
    STRFTIME('%m', [Order Date]) AS Month,
    SUM(Sales) / COUNT(DISTINCT [Order ID]) AS Avg_Order_Value
FROM orders
GROUP BY Year, Month
ORDER BY Year, Month
"""
avg_order_value = pd.read_sql_query(query2, con)
avg_order_value


,Year,Month,Avg_Order_Value
0,2014,01,444.902969
1,2014,02,161.424714
2,2014,03,784.380408
3,2014,04,428.717348
4,2014,05,342.728797
5,2014,06,524.168600
6,2014,07,522.252200
7,2014,08,387.631507
8,2014,09,629.056545
9,2014,10,403.248628


## Monthly Revenue Trend for a Specific Year (2017)

In [94]:
query3 = """
SELECT 
    STRFTIME('%m', [Order Date]) AS Month,
    SUM(Sales) AS Monthly_Revenue
FROM orders
WHERE STRFTIME('%Y', [Order Date]) = '2017'
GROUP BY Month
ORDER BY Month
"""
revenue_2017 = pd.read_sql_query(query3, con)
revenue_2017


,Month,Monthly_Revenue
0,01,43971.3740
1,02,20301.1334
2,03,58872.3528
3,04,36521.5361
4,05,44261.1102
5,06,52981.7257
6,07,45264.4160
7,08,63120.8880
8,09,87866.6520
9,10,77776.9232


## Top 3 Months with Highest Revenue

In [97]:
query4 = """
SELECT 
    STRFTIME('%Y-%m', [Order Date]) AS Year_Month,
    SUM(Sales) AS Monthly_Revenue
FROM orders
GROUP BY Year_Month
ORDER BY Monthly_Revenue DESC
LIMIT 3
"""
top_months = pd.read_sql_query(query4, con)
top_months


,Year_Month,Monthly_Revenue
0,2017-11,118447.825
1,2016-12,96999.043
2,2017-09,87866.652


## Quarter-wise Revenue Trend

In [100]:
query5 = """
SELECT 
    STRFTIME('%Y', [Order Date]) AS Year,
    'Q' || ((CAST(STRFTIME('%m', [Order Date]) AS INTEGER) - 1) / 3 + 1) AS Quarter,
    SUM(Sales) AS Total_Revenue
FROM orders
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
quarterly_summary = pd.read_sql_query(query5, con)
quarterly_summary


,Year,Quarter,Total_Revenue
0,2014,Q1,74447.7960
1,2014,Q2,86538.7596
2,2014,Q3,143633.2123
3,2014,Q4,179627.7302
4,2015,Q1,68851.7386
5,2015,Q2,89124.1870
6,2015,Q3,130259.5752
7,2015,Q4,182297.0082
8,2016,Q1,93237.1810
9,2016,Q2,136082.3010


## Monthly Revenue Trend by Region

In [103]:
query = """
SELECT 
    STRFTIME('%Y-%m', [Order Date]) AS Year_Month,
    Region,
    SUM(Sales) AS Revenue
FROM orders
GROUP BY Region, Year_Month
ORDER BY Region, Year_Month
"""
monthly_region = pd.read_sql_query(query, con)
monthly_region

,Year_Month,Region,Revenue
0,2014-01,Central,1539.906
1,2014-02,Central,1233.174
2,2014-03,Central,5827.602
3,2014-04,Central,3712.340
4,2014-05,Central,4048.506
...,...,...,...
187,2017-08,West,25737.894
188,2017-09,West,27907.037
189,2017-10,West,21212.436
190,2017-11,West,28941.787


## Yearly Revenue Trend by Category

In [106]:
query = """
SELECT 
    STRFTIME('%Y', [Order Date]) AS Year,
    Category,
    SUM(Sales) AS Revenue
FROM orders
GROUP BY Year, Category
ORDER BY Year, Category
"""
yearly_category = pd.read_sql_query(query, con)
yearly_category


,Year,Category,Revenue
0,2014,Furniture,157192.8531
1,2014,Office Supplies,151776.4120
2,2014,Technology,175278.2330
3,2015,Furniture,170518.2370
4,2015,Office Supplies,137233.4630
5,2015,Technology,162780.8090
6,2016,Furniture,198901.4360
7,2016,Office Supplies,183939.9820
8,2016,Technology,226364.1800
9,2017,Furniture,215387.2692


## Quarterly Order Volume by Sub-Category

In [109]:
query = """
SELECT 
    STRFTIME('%Y', [Order Date]) AS Year,
    'Q' || ((CAST(STRFTIME('%m', [Order Date]) AS INTEGER) - 1) / 3 + 1) AS Quarter,
    [Sub-Category],
    COUNT(DISTINCT [Order ID]) AS Total_Orders
FROM orders
GROUP BY Year, Quarter, [Sub-Category]
ORDER BY Year, Quarter, [Sub-Category]
"""
quarterly_subcat = pd.read_sql_query(query, con)
quarterly_subcat


,Year,Quarter,Sub-Category,Total_Orders
0,2014,Q1,Accessories,16
1,2014,Q1,Appliances,9
2,2014,Q1,Art,29
3,2014,Q1,Binders,38
4,2014,Q1,Bookcases,8
...,...,...,...,...
266,2017,Q4,Paper,146
267,2017,Q4,Phones,100
268,2017,Q4,Storage,101
269,2017,Q4,Supplies,16


## Monthly Average Sales per Order by Ship Mode

In [112]:
query = """
SELECT 
    STRFTIME('%Y-%m', [Order Date]) AS Year_Month,
    [Ship Mode],
    SUM(Sales) / COUNT(DISTINCT [Order ID]) AS Avg_Sale_Per_Order
FROM orders
GROUP BY Year_Month, [Ship Mode]
ORDER BY Year_Month, [Ship Mode]
"""
monthly_shipmode = pd.read_sql_query(query, con)
monthly_shipmode


,Year_Month,Ship Mode,Avg_Sale_Per_Order
0,2014-01,First Class,150.932857
1,2014-01,Second Class,373.786000
2,2014-01,Standard Class,575.665737
3,2014-02,First Class,168.783200
4,2014-02,Same Day,25.160000
...,...,...,...
183,2017-11,Standard Class,442.437800
184,2017-12,First Class,386.936500
185,2017-12,Same Day,240.675077
186,2017-12,Second Class,419.760154


## Revenue Growth Rate Year-over-Year

In [129]:
query = """
SELECT 
    STRFTIME('%Y', [Order Date]) AS Year,
    SUM(Sales) AS Total_Revenue
FROM orders
GROUP BY Year
ORDER BY Year
"""
revenue_yearly = pd.read_sql_query(query, con)
revenue_yearly

,Year,Total_Revenue
0,2014,484247.4981
1,2015,470532.5090
2,2016,609205.5980
3,2017,733215.2552


## Number of Orders by Week

In [135]:
query = """
SELECT 
    STRFTIME('%Y-%W', [Order Date]) AS Year_Week,
    COUNT(DISTINCT [Order ID]) AS Total_Orders
FROM orders
GROUP BY Year_Week
ORDER BY Year_Week
"""
weekly_orders = pd.read_sql_query(query, con)
weekly_orders

,Year_Week,Total_Orders
0,2014-00,3
1,2014-01,7
2,2014-02,9
3,2014-03,9
4,2014-04,7
...,...,...
207,2017-48,62
208,2017-49,62
209,2017-50,37
210,2017-51,52


## Monthly Revenue by Segment

In [137]:
query = """
SELECT 
    STRFTIME('%Y-%m', [Order Date]) AS Year_Month,
    Segment,
    SUM(Sales) AS Revenue
FROM orders
GROUP BY Year_Month, Segment
ORDER BY Year_Month, Segment
"""
monthly_segment = pd.read_sql_query(query, con)
monthly_segment

,Year_Month,Segment,Revenue
0,2014-01,Consumer,6927.8170
1,2014-01,Corporate,1701.5280
2,2014-01,Home Office,5607.5500
3,2014-02,Consumer,3167.8540
4,2014-02,Corporate,1183.6680
...,...,...,...
139,2017-11,Corporate,44644.0762
140,2017-11,Home Office,24013.6840
141,2017-12,Consumer,50232.4558
142,2017-12,Corporate,20524.4320


## Monthly Number of Customers (based on unique Customer IDs)

In [139]:
query = """
SELECT 
    STRFTIME('%Y-%m', [Order Date]) AS Year_Month,
    COUNT(DISTINCT [Customer ID]) AS Unique_Customers
FROM orders
GROUP BY Year_Month
ORDER BY Year_Month
"""
monthly_customers = pd.read_sql_query(query, con)
monthly_customers

,Year_Month,Unique_Customers
0,2014-01,32
1,2014-02,27
2,2014-03,69
3,2014-04,64
4,2014-05,67
5,2014-06,63
6,2014-07,65
7,2014-08,70
8,2014-09,118
9,2014-10,75


In [141]:
con.commit()

In [143]:
con.close()